## Imports

In [36]:
import pandas as pd
import numpy as np
import os

## Data

In [37]:
path = os.path.join('..', 'data', 'processed', 'Lisboa_freguesias_preco_m2_wide.csv')
df = pd.read_csv(path)

In [38]:
if os.path.exists(path):
    df = pd.read_csv(path)
else:
    print(f"Erro: O ficheiro não foi encontrado em {os.path.abspath(path)}")

df.head()

,Data,Ajuda,Alcântara,Alvalade,Areeiro,Arroios,Avenidas Novas,Beato,Belém,Benfica,...,Marvila,Misericórdia,Olivais,Parque das Nações,Penha de França,Santa Clara,Santa Maria Maior,Santo António,São Domingos de Benfica,São Vicente
0,May/2015,NaN,NaN,NaN,NaN,9.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Jun/2015,NaN,NaN,8.6,NaN,9.1,10.7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Jul/2015,NaN,NaN,9.5,NaN,8.9,9.3,NaN,9.7,NaN,...,NaN,NaN,NaN,11.9,NaN,NaN,NaN,NaN,8.9,NaN
3,Aug/2015,NaN,NaN,9.7,NaN,8.1,9.6,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,NaN
4,Sep/2015,NaN,NaN,10.6,NaN,8.1,10.4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,NaN


In [39]:
df.rename(columns={df.columns[0]:'date'}, inplace=True)

In [40]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values('date').reset_index(drop=True)

C:\Users\migue\AppData\Local\Temp\ipykernel_25352\1729161540.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


In [41]:
fregs = [c for c in df.columns if c != 'date']

In [42]:
summary = []
for f in fregs:
    series = df[f]
    total = len(series)
    n_missing = series.isna().sum()
    pct_missing = 100 * n_missing / total
    mask = series.isna().astype(int).values
    max_run = 0
    cur = 0
    for v in mask:
        if v==1:
            cur += 1
            if cur > max_run: max_run = cur
        else:
            cur = 0

    if series.dropna().shape[0] > 0:
        first_valid = df.loc[series.first_valid_index(), 'date'].strftime('%Y-%m')
        last_valid = df.loc[series.last_valid_index(), 'date'].strftime('%Y-%m')
    else:
        first_valid = None
        last_valid = None

    summary.append({
        'freguesia': f,
        'total_months': total,
        'n_missing': int(n_missing),
        'pct_missing': round(pct_missing,2),
        'max_consecutive_NA_months': int(max_run),
        'first_valid_ym': first_valid,
        'last_valid_ym': last_valid
    })

In [43]:
summary_df = pd.DataFrame(summary).sort_values('pct_missing')
summary_df

,freguesia,total_months,n_missing,pct_missing,max_consecutive_NA_months,first_valid_ym,last_valid_ym
4,Arroios,128,0,0.00,0,2015-05,2025-12
2,Alvalade,128,1,0.78,1,2015-06,2025-12
5,Avenidas Novas,128,1,0.78,1,2015-06,2025-12
13,Lumiar,128,1,0.78,1,2015-06,2025-12
22,São Domingos de Benfica,128,2,1.56,2,2015-07,2025-12
17,Parque das Nações,128,4,3.12,2,2015-07,2025-12
21,Santo António,128,5,3.91,5,2015-10,2025-12
12,Estrela,128,6,4.69,2,2015-07,2025-12
3,Areeiro,128,6,4.69,6,2015-11,2025-12
8,Benfica,128,7,5.47,6,2015-11,2025-12


Data Quality Analysis - Missing Values by Parish

- Parishes with near-zero missing values (NAs): Arroios, Alvalade, Avenidas Novas, Lumiar (pct ~0–0.8%).

- Parishes with few scattered NAs (pct <12%): Santo António, Estrela, Areeiro, Benfica, Campolide, Belém, Penha de França, Misericórdia, Santa Maria Maior, São Vicente, Campo de Ourique — these show low to moderate missing percentages.

- Parishes with high structural missingness (pct >> 40%): Santa Clara (41.7%), Beato (43.3%), Alcântara (74%), Marvila (75.6%), Carnide (75.6%) — these are clear cases of significant structural data gaps.

- Special Cases: Ajuda (30% missing; first_valid_ym 2017-04): No data available for the 2015–2016 period.

    Olivais (27.6% missing; first_valid_ym 2018-02): Both cases show long initial blocks without data, indicating a later start in data collection.

- max_consecutive_NA_months identifies long missing data blocks.

    Examples: Alcântara (94 consecutive months), Carnide/Marvila (96), Beato (42), Olivais (33), Ajuda (23), Santa Clara (21), Santa Maria Maior/Misericórdia/São Vicente (15), Campo de Ourique (9), Penha de França (13).

In [44]:
def get_na_blocks(mask):
    blocks = []
    cur = 0
    for v in mask:
        if v:
            cur += 1
        else:
            if cur>0:
                blocks.append(cur)
            cur = 0
    if cur>0: blocks.append(cur)
    return blocks


metrics = []
total_months = df.shape[0]
first_24_dates = df['date'].iloc[:24]

for f in fregs:
    series = df[f]
    mask = series.isna().values
    n_missing = int(mask.sum())
    pct_missing = 100 * n_missing / total_months

    blocks = get_na_blocks(mask)
    n_blocks = len(blocks)
    avg_block_len = np.mean(blocks) if n_blocks>0 else 0
    max_block = max(blocks) if n_blocks>0 else 0

    # Analysis of long gaps (>= 6 months)
    n_blocks_gt6 = sum(1 for b in blocks if b>=6)
    long_gap_total = sum([b for b in blocks if b>=6])
    long_gap_ratio = (long_gap_total / n_missing) if n_missing>0 else 0

    # Late start detection (first 24 months)
    first24_mask = df[f].iloc[:24].isna().sum()
    pct_missing_first24 = 100 * first24_mask / 24

    # Dispersion metric: n_blocks / n_missing 
    # High -> scattered small blocks; Low -> few concentrated long blocks
    dispersion = (n_blocks / n_missing) if n_missing>0 else np.nan
    if series.dropna().shape[0] > 0:
        first_valid = df.loc[series.first_valid_index(),'date']
        last_valid = df.loc[series.last_valid_index(),'date']
    else:
        first_valid = pd.NaT
        last_valid = pd.NaT

    metrics.append({
        'freguesia': f,
        'total_months': total_months,
        'n_missing': n_missing,
        'pct_missing': round(pct_missing, 2), # Total percentage of missing months
        'n_na_blocks': n_blocks, # Number of NA blocks (count of contiguous segments with 1+ NAs)
        'avg_block_length': round(avg_block_len, 2), # Average length of NA segments
        'max_consec_na': int(max_block), # Length of the longest consecutive NA block
        'n_long_blocks_gt6': n_blocks_gt6, # Count of NA blocks lasting 6 months or more
        'long_gap_ratio': round(long_gap_ratio, 2), # Proportion of missingness concentrated in long blocks (blocks >= 6)
        'pct_missing_start_24m': round(pct_missing_first24, 2), # % missing in first 24 months (indicates late data collection start)
        'dispersion_index': round(dispersion, 3), # Metric: n_blocks / n_missing (High = scattered; Low = concentrated)
        'first_valid_obs': first_valid, # Date of the first valid observation
        'last_valid_obs': last_valid # Date of the last valid observation
    })

metrics_df = pd.DataFrame(metrics).sort_values('pct_missing')
metrics_df

,freguesia,total_months,n_missing,pct_missing,n_na_blocks,avg_block_length,max_consec_na,n_long_blocks_gt6,long_gap_ratio,pct_missing_start_24m,dispersion_index,first_valid_obs,last_valid_obs
4,Arroios,128,0,0.00,0,0.00,0,0,0.00,0.00,NaN,2015-05-01,2025-12-01
2,Alvalade,128,1,0.78,1,1.00,1,0,0.00,4.17,1.000,2015-06-01,2025-12-01
5,Avenidas Novas,128,1,0.78,1,1.00,1,0,0.00,4.17,1.000,2015-06-01,2025-12-01
13,Lumiar,128,1,0.78,1,1.00,1,0,0.00,4.17,1.000,2015-06-01,2025-12-01
22,São Domingos de Benfica,128,2,1.56,1,2.00,2,0,0.00,8.33,0.500,2015-07-01,2025-12-01
17,Parque das Nações,128,4,3.12,2,2.00,2,0,0.00,16.67,0.500,2015-07-01,2025-12-01
21,Santo António,128,5,3.91,1,5.00,5,0,0.00,20.83,0.200,2015-10-01,2025-12-01
12,Estrela,128,6,4.69,4,1.50,2,0,0.00,25.00,0.667,2015-07-01,2025-12-01
3,Areeiro,128,6,4.69,1,6.00,6,1,1.00,25.00,0.167,2015-11-01,2025-12-01
8,Benfica,128,7,5.47,2,3.50,6,1,0.86,25.00,0.286,2015-11-01,2025-12-01


Missing Data Breakdown by Parish
Baseline: Considering the most complete series begins in May 2015 (anchored by Arroios):

1. Minimal Missingness (pct ~0–0.8%)
    - Arroios: No missing values. Range: May 2015 to Nov 2025.
    - Alvalade: 1 missing value. Range: June 2015 to Nov 2025.
    - Avenidas Novas: 1 missing value. Range: June 2015 to Nov 2025.
    - Lumiar: 1 missing value. Range: June 2015 to Nov 2025.

2. Low to Moderate Missingness (1.57% < pct < 7.87%)
    - São Domingos de Benfica: 2 missing values. Range: July 2015 to Nov 2025.
    - Parque das Nações: 4 missing values. May/June 2015 (Missing); July 2015 (Available); Aug/Sept 2015 (Missing). Continuous data from Oct 2015 to Nov 2025.
    - Santo António: 5 missing values. Range: Oct 2015 to Nov 2025.
    - Estrela: 6 missing values. May/June 2015 (Missing); July/Aug 2015 (Available); Sept 2015 (Missing); Oct/Nov 2015 (Available); Dec 2015 to Jan 2016 (Missing); Feb 2016 (Available); March 2016 (Missing). Continuous data from April 2016 to Nov 2025.
    - Areeiro: 6 missing values. Range: Nov 2015 to Nov 2025.
    - Benfica: 7 missing values. 6 missing at the start (Nov 2015 range) + 1 missing in Jan 2018.
    - Campo de Ourique: 9 missing values. Range: Feb 2016 to Nov 2025.
    - Campolide: 10 missing values. One block from May to Oct 2015 (6 values) + one block from April to July 2016 (4 values).
    - Belém: 10 missing values. Multiple blocks: May–June 2015 (2); Aug–Oct 2015 (3); April 2016 (1); Aug–Sept 2016 (2); Sept–Oct 2025 (2).

3. Intermediate Missingness (10.24% < pct < 11.81%)
    - Penha de França: 13 missing values. Range: June 2016 to Nov 2025.
    - Misericórdia: 15 missing values. Range: Aug 2016 to Nov 2025.
    - Santa Maria Maior: 15 missing values. Range: Aug 2016 to Nov 2025.
    - São Vicente: 15 missing values. Range: Aug 2016 to Nov 2025.

4. High Missingness / Late Starts (23.62% < pct < 27.56%)
    - Ajuda: 30 missing values. May 2015 to May 2017 gap (23); Sept 2017 (1); Nov 2017 to Feb 2018 (4); June to July 2018 (2).
    - Olivais: 35 missing values. Main block from May 2015 to Jan 2018 (33) + isolated gaps in July and Sept 2018.

5. Significant Structural Gaps (~43% pct)
    - Santa Clara: 53 missing values. Intermittent blocks: May 2015–May 2016 (13); Sept 2016 (1); Nov 2016–March 2017 (5); May 2017–Jan 2019 (21); Jan 2020 (1); Oct 2022–Sept 2023 (12).
    - Beato: 55 missing values. Initial structural block from May 2015 to Oct 2018 (42) + June 2019 (1) + Aug 2022 to July 2023 (12).

6. Critical Structural Gaps (74% < pct < 76%)
    - Alcântara: 94 missing values. Series effectively begins in March 2023.
    - Marvila: 96 missing values. Series effectively begins in May 2023.
    - Carnide: 96 missing values. Series effectively begins in May 2023.

In [45]:
df.head()

,date,Ajuda,Alcântara,Alvalade,Areeiro,Arroios,Avenidas Novas,Beato,Belém,Benfica,...,Marvila,Misericórdia,Olivais,Parque das Nações,Penha de França,Santa Clara,Santa Maria Maior,Santo António,São Domingos de Benfica,São Vicente
0,2015-05-01,NaN,NaN,NaN,NaN,9.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-06-01,NaN,NaN,8.6,NaN,9.1,10.7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-07-01,NaN,NaN,9.5,NaN,8.9,9.3,NaN,9.7,NaN,...,NaN,NaN,NaN,11.9,NaN,NaN,NaN,NaN,8.9,NaN
3,2015-08-01,NaN,NaN,9.7,NaN,8.1,9.6,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,NaN
4,2015-09-01,NaN,NaN,10.6,NaN,8.1,10.4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,NaN


In [46]:
## exclude Santa Clara, Beato, Ajuda, olivais, Alcântara, Marvila, Carnide
exclude_fregs = ['Santa Clara', 'Beato', 'Ajuda', 'Olivais', 'Alcântara', 'Marvila', 'Carnide']
df = df[[c for c in df.columns if c not in exclude_fregs]]
df.head()

,date,Alvalade,Areeiro,Arroios,Avenidas Novas,Belém,Benfica,Campo de Ourique,Campolide,Estrela,Lumiar,Misericórdia,Parque das Nações,Penha de França,Santa Maria Maior,Santo António,São Domingos de Benfica,São Vicente
0,2015-05-01,NaN,NaN,9.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-06-01,8.6,NaN,9.1,10.7,NaN,NaN,NaN,NaN,NaN,8.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-07-01,9.5,NaN,8.9,9.3,9.7,NaN,NaN,NaN,9.0,8.8,NaN,11.9,NaN,NaN,NaN,8.9,NaN
3,2015-08-01,9.7,NaN,8.1,9.6,NaN,NaN,NaN,NaN,8.9,8.9,NaN,NaN,NaN,NaN,NaN,8.5,NaN
4,2015-09-01,10.6,NaN,8.1,10.4,NaN,NaN,NaN,NaN,NaN,8.8,NaN,NaN,NaN,NaN,NaN,8.5,NaN


In [47]:
## start df at 2016-08-01 date and end in 2025-11-01
df = df[(df['date'] >= '2016-07-01') & (df['date'] <= '2025-11-01')].reset_index(drop=True)
df.head()

,date,Alvalade,Areeiro,Arroios,Avenidas Novas,Belém,Benfica,Campo de Ourique,Campolide,Estrela,Lumiar,Misericórdia,Parque das Nações,Penha de França,Santa Maria Maior,Santo António,São Domingos de Benfica,São Vicente
0,2016-07-01,10.9,10.8,11.0,11.3,9.9,8.4,12.6,NaN,11.4,8.9,NaN,12.3,10.0,NaN,12.7,10.0,NaN
1,2016-08-01,11.1,11.1,11.8,12.1,NaN,8.9,11.2,11.0,12.5,9.2,17.5,12.7,10.3,18.0,13.1,10.6,12.0
2,2016-09-01,10.8,10.8,11.7,12.2,NaN,9.3,11.2,11.8,13.0,9.2,20.0,12.9,10.6,20.0,13.2,10.5,15.0
3,2016-10-01,10.2,11.5,12.2,12.2,11.5,9.6,11.7,12.0,13.3,9.6,20.0,12.7,10.7,21.1,13.6,11.0,15.6
4,2016-11-01,10.8,11.6,12.9,12.1,11.5,9.0,12.2,13.6,13.3,10.2,20.0,12.7,10.0,23.9,15.0,10.8,17.5


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113 entries, 0 to 112
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     113 non-null    datetime64[ns]
 1   Alvalade                 113 non-null    float64       
 2   Areeiro                  113 non-null    float64       
 3   Arroios                  113 non-null    float64       
 4   Avenidas Novas           113 non-null    float64       
 5   Belém                    109 non-null    float64       
 6   Benfica                  112 non-null    float64       
 7   Campo de Ourique         113 non-null    float64       
 8   Campolide                112 non-null    float64       
 9   Estrela                  113 non-null    float64       
 10  Lumiar                   113 non-null    float64       
 11  Misericórdia             112 non-null    float64       
 12  Parque das Nações        113 non-nul

In [49]:
## missing values analysis after filtering
missing_values = df.isna().sum()
missing_values

date                       0
Alvalade                   0
Areeiro                    0
Arroios                    0
Avenidas Novas             0
Belém                      4
Benfica                    1
Campo de Ourique           0
Campolide                  1
Estrela                    0
Lumiar                     0
Misericórdia               1
Parque das Nações          0
Penha de França            0
Santa Maria Maior          1
Santo António              0
São Domingos de Benfica    0
São Vicente                1
dtype: int64

## Imputation of missing values

In [50]:
def simple_linear_imputation(df, date_col='date'):
    """
    Missing values:
    - Belém: Aug 2016, Sep 2016, Sep 2025, Oct 2025
    - Benfica: Jan 2018
    
    Returns:
    --------
    df_imputed : DataFrame with imputed values
    df_flags : DataFrame with imputation flags (same structure)
    imputation_report : DataFrame with details of imputed values
    """
    
    df_imputed = df.copy()
    df_imputed[date_col] = pd.to_datetime(df_imputed[date_col])
    df_imputed = df_imputed.sort_values(date_col)
    df_imputed = df_imputed.set_index(date_col)
    

    df_flags = df_imputed.isna().astype(int)
    df_flags.columns = [col + '_imputed_flag' for col in df_flags.columns]
    
    imputation_details = []
    
    print("="*80)
    print("LINEAR INTERPOLATION - MISSING VALUES")
    print("="*80)
    
    # Identify and document missing values before interpolation
    for col in df_imputed.columns:
        missing_dates = df_imputed[df_imputed[col].isna()].index
        
        if len(missing_dates) > 0:
            print(f"\n{col}: {len(missing_dates)} missing")
            
            for date in missing_dates:
                # Get before and after values for documentation
                idx_pos = df_imputed.index.get_loc(date)
                
                value_before = np.nan
                value_after = np.nan
                date_before = None
                date_after = None
                
                # Look backward for last observed value
                for i in range(idx_pos - 1, -1, -1):
                    if pd.notna(df_imputed.iloc[i][col]):
                        value_before = df_imputed.iloc[i][col]
                        date_before = df_imputed.index[i]
                        break
                
                # Look forward for next observed value
                for i in range(idx_pos + 1, len(df_imputed)):
                    if pd.notna(df_imputed.iloc[i][col]):
                        value_after = df_imputed.iloc[i][col]
                        date_after = df_imputed.index[i]
                        break
                
                imputation_details.append({
                    'parish': col,
                    'date': date.strftime('%Y-%m'),
                    'date_before': date_before.strftime('%Y-%m') if date_before else None,
                    'value_before': value_before,
                    'date_after': date_after.strftime('%Y-%m') if date_after else None,
                    'value_after': value_after
                })
                
                print(f"  {date.strftime('%Y-%m')}: "
                      f"between {date_before.strftime('%Y-%m') if date_before else 'START'} "
                      f"({value_before:.2f} €/m²) and "
                      f"{date_after.strftime('%Y-%m') if date_after else 'END'} "
                      f"({value_after:.2f} €/m²)")
    
    # Perform linear interpolation
    # limit_direction='both' handles boundary cases (start and end of series)
    df_imputed = df_imputed.interpolate(method='linear', limit_direction='both', axis=0)
    
    # Add imputed values to the report
    for detail in imputation_details:
        parish = detail['parish']
        date = pd.to_datetime(detail['date'])
        detail['imputed_value'] = df_imputed.loc[date, parish]
    
    # Reset index to get date back as column
    df_imputed = df_imputed.reset_index()
    df_flags = df_flags.reset_index()
    
    # Summary
    total_imputed = df_flags.filter(like='_imputed_flag').sum().sum()
    total_obs = len(df_imputed) * (len(df_imputed.columns) - 1)  # -1 for date column
    
    print("\n" + "="*80)
    print("IMPUTATION COMPLETE")
    print("="*80)
    print(f"Total imputed: {total_imputed} observations")
    print(f"Total observations: {total_obs}")
    print(f"Percentage imputed: {total_imputed/total_obs*100:.3f}%")
    
    print("\nImputed by parish:")
    for col in df_flags.columns:
        if '_imputed_flag' in col:
            count = df_flags[col].sum()
            if count > 0:
                parish = col.replace('_imputed_flag', '')
                print(f"  {parish}: {count}")
    
    # Create imputation report DataFrame
    imputation_report = pd.DataFrame(imputation_details)
    
    # Reorder columns for clarity
    if len(imputation_report) > 0:
        imputation_report = imputation_report[[
            'parish', 'date', 
            'date_before', 'value_before',
            'imputed_value',
            'date_after', 'value_after'
        ]]
    
    print("\n" + "="*80)
    print("IMPUTATION REPORT - DETAILED VALUES")
    print("="*80)
    print(imputation_report.to_string(index=False))
    print("="*80)
    
    return df_imputed, df_flags, imputation_report

In [51]:
def verify_imputation(df_original, df_imputed, df_flags):
    """
    Verify that imputation worked correctly.
    """
    print("\n" + "="*80)
    print("VERIFICATION")
    print("="*80)
    
    # Check: exactly 5 values imputed
    total_flags = df_flags.filter(like='_imputed_flag').sum().sum()
    print(f"\n✓ Total flagged as imputed: {total_flags}")
    assert total_flags == 5, f"ERROR: Expected 5, got {total_flags}"
    
    # Check: no missing values remain
    price_cols = [col for col in df_imputed.columns if col != 'date']
    n_missing = df_imputed[price_cols].isna().sum().sum()
    print(f"✓ Remaining missing values: {n_missing}")
    assert n_missing == 0, f"ERROR: Still have {n_missing} missing values!"
    
    # Check: specific expected locations
    expected_missing = [
        ('Belém', '2016-08-01'),
        ('Belém', '2016-09-01'),
        ('Benfica', '2018-01-01'),
        ('Belém', '2025-09-01'),
        ('Belém', '2025-10-01')
    ]
    
    print("\n✓ Verifying exact locations:")
    for parish, date in expected_missing:
        date_mask = df_imputed['date'] == date
        value = df_imputed.loc[date_mask, parish].values[0]
        flag = df_flags.loc[date_mask, parish + '_imputed_flag'].values[0]
        
        print(f"  {parish} {date}: flag={flag}, value={value:.2f} €/m²")
        assert flag == 1, f"ERROR: {parish} {date} not flagged!"
        assert pd.notna(value), f"ERROR: {parish} {date} still NaN!"
    
    # Check: original values unchanged
    print("\n✓ Checking original values unchanged...")
    df_orig_indexed = df_original.set_index('date')
    df_imp_indexed = df_imputed.set_index('date')
    
    for col in price_cols:
        # Get indices where original had values
        orig_not_null = df_orig_indexed[col].notna()
        
        if orig_not_null.sum() > 0:
            # Compare values at those indices
            orig_vals = df_orig_indexed.loc[orig_not_null, col]
            imp_vals = df_imp_indexed.loc[orig_not_null, col]
            
            max_diff = (orig_vals - imp_vals).abs().max()
            
            if max_diff > 1e-10:
                print(f"  WARNING: {col} changed by {max_diff:.2e}")
            else:
                print(f"  {col}: no changes ✓")
    
    print("\n" + "="*80)
    print("ALL VERIFICATION CHECKS PASSED ✓")
    print("="*80)

In [52]:
def convert_to_long_format(df_imputed, df_flags, date_col='date'):
    """
    Convert wide format to long format for ML modeling.
    
    Returns:
    --------
    DataFrame with columns: date | parish | price_m2 | imputed_flag
    """
    
    # Melt price data
    df_long = df_imputed.melt(
        id_vars=[date_col],
        var_name='parish',
        value_name='price_m2'
    )
    
    # Melt flag data
    df_flags_long = df_flags.melt(
        id_vars=[date_col],
        var_name='parish_flag',
        value_name='imputed_flag'
    )
    
    # Remove '_imputed_flag' suffix
    df_flags_long['parish'] = df_flags_long['parish_flag'].str.replace('_imputed_flag', '')
    df_flags_long = df_flags_long.drop('parish_flag', axis=1)
    
    # Merge
    df_final = df_long.merge(
        df_flags_long,
        on=[date_col, 'parish'],
        how='left'
    )
    
    # Sort
    df_final = df_final.sort_values([date_col, 'parish']).reset_index(drop=True)
    
    print(f"\nConverted to long format: {len(df_final)} rows")
    print(f"Structure: {df_final.columns.tolist()}")
    
    return df_final

In [53]:
df_imputed, df_flags, report = simple_linear_imputation(df, date_col='date')

LINEAR INTERPOLATION - MISSING VALUES

Belém: 4 missing
  2016-08: between 2016-07 (9.90 €/m²) and 2016-10 (11.50 €/m²)
  2016-09: between 2016-07 (9.90 €/m²) and 2016-10 (11.50 €/m²)
  2025-09: between 2025-08 (19.60 €/m²) and 2025-11 (21.40 €/m²)
  2025-10: between 2025-08 (19.60 €/m²) and 2025-11 (21.40 €/m²)

Benfica: 1 missing
  2018-01: between 2017-12 (10.20 €/m²) and 2018-02 (12.70 €/m²)

Campolide: 1 missing
  2016-07: between START (nan €/m²) and 2016-08 (11.00 €/m²)

Misericórdia: 1 missing
  2016-07: between START (nan €/m²) and 2016-08 (17.50 €/m²)

Santa Maria Maior: 1 missing
  2016-07: between START (nan €/m²) and 2016-08 (18.00 €/m²)

São Vicente: 1 missing
  2016-07: between START (nan €/m²) and 2016-08 (12.00 €/m²)

IMPUTATION COMPLETE
Total imputed: 9 observations
Total observations: 1921
Percentage imputed: 0.469%

Imputed by parish:
  Belém: 4
  Benfica: 1
  Campolide: 1
  Misericórdia: 1
  Santa Maria Maior: 1
  São Vicente: 1

IMPUTATION REPORT - DETAILED VALUES

In [54]:
report

,parish,date,date_before,value_before,imputed_value,date_after,value_after
0,Belém,2016-08,2016-07,9.9,10.433333,2016-10,11.5
1,Belém,2016-09,2016-07,9.9,10.966667,2016-10,11.5
2,Belém,2025-09,2025-08,19.6,20.200000,2025-11,21.4
3,Belém,2025-10,2025-08,19.6,20.800000,2025-11,21.4
4,Benfica,2018-01,2017-12,10.2,11.450000,2018-02,12.7
5,Campolide,2016-07,None,NaN,11.000000,2016-08,11.0
6,Misericórdia,2016-07,None,NaN,17.500000,2016-08,17.5
7,Santa Maria Maior,2016-07,None,NaN,18.000000,2016-08,18.0
8,São Vicente,2016-07,None,NaN,12.000000,2016-08,12.0


In [55]:
df_imputed

,date,Alvalade,Areeiro,Arroios,Avenidas Novas,Belém,Benfica,Campo de Ourique,Campolide,Estrela,Lumiar,Misericórdia,Parque das Nações,Penha de França,Santa Maria Maior,Santo António,São Domingos de Benfica,São Vicente
0,2016-07-01,10.9,10.8,11.0,11.3,9.900000,8.4,12.6,11.0,11.4,8.9,17.5,12.3,10.0,18.0,12.7,10.0,12.0
1,2016-08-01,11.1,11.1,11.8,12.1,10.433333,8.9,11.2,11.0,12.5,9.2,17.5,12.7,10.3,18.0,13.1,10.6,12.0
2,2016-09-01,10.8,10.8,11.7,12.2,10.966667,9.3,11.2,11.8,13.0,9.2,20.0,12.9,10.6,20.0,13.2,10.5,15.0
3,2016-10-01,10.2,11.5,12.2,12.2,11.500000,9.6,11.7,12.0,13.3,9.6,20.0,12.7,10.7,21.1,13.6,11.0,15.6
4,2016-11-01,10.8,11.6,12.9,12.1,11.500000,9.0,12.2,13.6,13.3,10.2,20.0,12.7,10.0,23.9,15.0,10.8,17.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,2025-07-01,20.0,18.6,22.7,21.4,18.900000,17.7,22.1,21.8,24.7,16.3,27.3,19.1,20.2,28.0,28.9,18.0,23.8
109,2025-08-01,20.0,19.1,23.4,21.3,19.600000,18.2,22.4,21.7,24.6,16.3,27.3,19.1,20.8,28.6,29.0,17.8,23.6
110,2025-09-01,20.4,19.9,23.2,21.9,20.200000,18.2,22.4,21.6,25.0,16.3,27.1,19.8,20.9,28.6,29.1,17.9,23.5
111,2025-10-01,20.9,20.0,24.0,22.7,20.800000,18.3,22.6,21.7,25.6,16.5,27.6,20.0,20.8,28.9,29.2,17.7,24.5


In [56]:
df_final = convert_to_long_format(df_imputed, df_flags, date_col='date')


Converted to long format: 1921 rows
Structure: ['date', 'parish', 'price_m2', 'imputed_flag']


In [57]:
# filter to start at 2016-08-01 and end at 2025-11-01
df_final = df_final[(df_final['date'] >= '2016-08-01') & (df_final['date'] <= '2025-11-01')].reset_index(drop=True)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1904 entries, 0 to 1903
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          1904 non-null   datetime64[ns]
 1   parish        1904 non-null   object        
 2   price_m2      1904 non-null   float64       
 3   imputed_flag  1904 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 59.6+ KB


In [58]:
df_final["Year"]=df_final['date'].dt.year
df_final["Month"]=df_final['date'].dt.month
df_final = df_final[['date', 'Year', 'Month', 'parish', 'price_m2', 'imputed_flag']]
df_final.head()

,date,Year,Month,parish,price_m2,imputed_flag
0,2016-08-01,2016,8,Alvalade,11.100000,0
1,2016-08-01,2016,8,Areeiro,11.100000,0
2,2016-08-01,2016,8,Arroios,11.800000,0
3,2016-08-01,2016,8,Avenidas Novas,12.100000,0
4,2016-08-01,2016,8,Belém,10.433333,1


In [60]:
# 1. Criar o Esqueleto Perfeito (O que DEVERÍAMOS ter)
# Usando as tuas datas de guilhotina: Agosto 2016 a Novembro 2025
theoretical_dates = pd.period_range(start='2016-08', end='2025-11', freq='M')
target_parishes = [c for c in df_final.columns if c != 'date'] # Parishes not excluded

index_mestre = pd.MultiIndex.from_product([theoretical_dates, target_parishes], names=['AnoMes', 'Freguesia'])
df_skeleton = pd.DataFrame(index=index_mestre).reset_index()
df_skeleton['AnoMes_Str'] = df_skeleton['AnoMes'].astype(str)

df_target_check = df_final.copy()
df_target_check['AnoMes_Str'] = pd.to_datetime(df_target_check['date']).dt.to_period('M').astype(str)

# Transform to Long Format (Melt) for row-by-row comparison
df_target_long = pd.melt(df_target_check, id_vars=['AnoMes_Str'], value_vars=target_parishes, 
                         var_name='Freguesia', value_name='Preco_m2')

# 3. IDENTIFICAR OS FANTASMAS (Onde o Urbanismo existe mas o Preço não)
df_comparison = pd.merge(df_skeleton, df_target_long, on=['AnoMes_Str', 'Freguesia'], how='left')
missing_data = df_comparison[df_comparison['Preco_m2'].isna()].copy()

# 4. RELATÓRIO DE DANOS
print(f"--- MISSING DATA AUDIT REPORT (IDEALISTA) ---")
print(f"Total Month-Parish combinations missing: {len(missing_data)}")

if len(missing_data) > 0:
    print("\n📍 Parishes with highest frequency of missing months:")
    print(missing_data['Parish'].value_counts())
    
    print("\n📅 Sample of detected gaps (First 10 entries):")
    print(missing_data[['YearMonth_Str', 'Parish']].head(10))
    
else:
    print("\nNo data gaps found between 2016-08 and 2025-11 for the selected parishes.")

--- MISSING DATA AUDIT REPORT (IDEALISTA) ---
Total Month-Parish combinations missing: 0

No data gaps found between 2016-08 and 2025-11 for the selected parishes.


In [61]:
output_path = os.path.join('..', 'data', 'processed', 'Lisboa_freguesias_preco_m2_long_imputed.csv')
df_final.to_csv(output_path, index=False)